In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [2]:
expression_data = pd.read_csv('expression_data.csv')
gene_variances = expression_data.var(axis=1)

def naive_bayes_train_and_predict(num_genes: int):
    top_n_genes = gene_variances.sort_values(ascending=False).head(num_genes).index
    top_variable_expression = expression_data.loc[top_n_genes].T

    X = top_variable_expression
    y = pd.read_csv('../data/processed/group_annotation.csv', index_col=0)['Group']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y
    )

    nb_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('nb', GaussianNB())
    ])

    param_grid = {
        'nb__var_smoothing': np.logspace(-10, -8, 10)  # Tune smoothing parameter
    }

    grid_search = GridSearchCV(
        estimator=nb_pipeline,
        param_grid=param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1  # Use all available cores
    )
    grid_search.fit(X_train, y_train)

    best_var_smoothing = grid_search.best_params_['nb__var_smoothing']

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    test_accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return best_var_smoothing, test_accuracy, report

### 1. Train Naive Bayes on Assignment 1 Groups

In [3]:
best_var_smoothing, test_accuracy, report = naive_bayes_train_and_predict(5000)

print(f"Assignment 1 groups prediction accuracy with 5000 genes and best var_smoothing={best_var_smoothing:.2e}: {test_accuracy:.4f}")
print("\nClassification report:")
print(report)

Assignment 1 groups prediction accuracy with 5000 genes and best var_smoothing=1.00e-10: 0.8889

Classification report:
              precision    recall  f1-score   support

          T0       0.87      0.93      0.90        14
          T3       0.92      0.85      0.88        13

    accuracy                           0.89        27
   macro avg       0.89      0.89      0.89        27
weighted avg       0.89      0.89      0.89        27



### 2. Train Naive Bayes on GMM Clusters

In [4]:
# load GMM clustering results
clusters = pd.read_csv('../results/cluster_results_gmm.csv').set_index('Sample')
label_col = 'Cluster_GMM'

# Use top 5000 genes for consistency
top_5000_genes = gene_variances.sort_values(ascending=False).head(5000).index
top_variable_expression = expression_data.loc[top_5000_genes].T

X = top_variable_expression.copy()
common = X.index.intersection(clusters.index)
if len(common) == 0:
    raise ValueError("No sample IDs match between top_variable_expression.index and cluster_results_gmm.csv Sample column.")
X = X.loc[common]
y = clusters.loc[common, label_col].astype(int)

# sanity check
print("Cluster counts:\n", y.value_counts())

# split
X_train_gmm, X_test_gmm, y_train_gmm, y_test_gmm = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

nb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

param_grid = {
    'nb__var_smoothing': np.logspace(-10, -8, 10)
}

grid_search_gmm = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_gmm.fit(X_train_gmm, y_train_gmm)

best_model_gmm = grid_search_gmm.best_estimator_
y_pred_gmm = best_model_gmm.predict(X_test_gmm)

print(f"\nBest var_smoothing for GMM clustering results classification: {grid_search_gmm.best_params_['nb__var_smoothing']:.2e}")

Cluster counts:
 Cluster_GMM
0    48
1    40
Name: count, dtype: int64

Best var_smoothing for GMM clustering results classification: 1.00e-10

Best var_smoothing for GMM clustering results classification: 1.00e-10


In [5]:
test_accuracy_gmm = accuracy_score(y_test_gmm, y_pred_gmm)
print(f"GMM clusters prediction accuracy: {test_accuracy_gmm:.4f}")

print("\nNaive Bayes classification report:")
print(classification_report(y_test_gmm, y_pred_gmm))
print("Confusion matrix:")
print(confusion_matrix(y_test_gmm, y_pred_gmm))
pd.DataFrame({'sample_id': X_test_gmm.index, 'true_cluster': y_test_gmm.values, 'pred_cluster_nb': y_pred_gmm}).to_csv('../results/naive_bayes_gmm_predictions.csv', index=False)

GMM clusters prediction accuracy: 1.0000

Naive Bayes classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         8

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18

Confusion matrix:
[[10  0]
 [ 0  8]]


### 3. Train Naive Bayes on Different Numbers of Genes

In [6]:
for n in [10, 100, 1000, 10000]:
    best_var_smoothing, test_accuracy, report = naive_bayes_train_and_predict(n)

    print(f"Assignment 1 groups prediction accuracy with {n} genes and best var_smoothing={best_var_smoothing:.2e}: {test_accuracy:.4f}")
    print("\nClassification report:")
    print(report)

Assignment 1 groups prediction accuracy with 10 genes and best var_smoothing=1.00e-10: 0.7407

Classification report:
              precision    recall  f1-score   support

          T0       0.77      0.71      0.74        14
          T3       0.71      0.77      0.74        13

    accuracy                           0.74        27
   macro avg       0.74      0.74      0.74        27
weighted avg       0.74      0.74      0.74        27

Assignment 1 groups prediction accuracy with 100 genes and best var_smoothing=1.00e-10: 0.8148

Classification report:
              precision    recall  f1-score   support

          T0       0.85      0.79      0.81        14
          T3       0.79      0.85      0.81        13

    accuracy                           0.81        27
   macro avg       0.82      0.82      0.81        27
weighted avg       0.82      0.81      0.81        27

Assignment 1 groups prediction accuracy with 1000 genes and best var_smoothing=1.00e-10: 0.7778

Classificati